# NB25 · Descriptors & Metaprogramming

`advanced/` · 20 questions · the last one is a story project

What happens when Python looks up, sets and creates things: attribute
lookup, descriptors (the machinery behind `property`, methods and
`classmethod`), `__init_subclass__`, building classes with `type()`,
metaclasses and customised `isinstance`. Every task is something real:
validated fields, plugin registries, a tiny ORM.

## You'll need

- NB24: context managers
- NB21: advanced OOP

## You'll practise

- Attribute lookup order; __getattr__ vs __getattribute__; __setattr__
- Descriptors: __get__, __set__, __set_name__; data vs non-data descriptors
- How property, methods and classmethod are built on descriptors
- __init_subclass__ and __class_getitem__
- Creating classes with type(); metaclasses (__new__, __call__)
- inspect, vars(), dir(); monkey-patching; why eval and exec are dangerous
- __instancecheck__, __subclasshook__ and ABC.register: how isinstance can be customised

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Real-life

A `Profile` class has a normal attribute `name = "Ada"`. Add a
`__getattr__` so that reading any attribute that does **not** exist returns
the text `"unknown"` instead of raising. Print `name` and `age` of a profile.

**Expected output:**

```text
Ada
unknown
```

<details><summary><b>Solution</b></summary>

```python
class Profile:
    name = "Ada"
    def __getattr__(self, attr):
        return "unknown"

p = Profile()
print(p.name)
print(p.age)
```

</details>

In [ ]:
# your code here


### Q02 · Output-match

Without a `class` statement, use `type()` to create a class `Point` with
class attributes `x = 0` and `y = 0`. Make an instance and print its `x`
and the class name.

**Expected output:**

```text
0 Point
```

<details><summary><b>Solution</b></summary>

```python
Point = type("Point", (), {"x": 0, "y": 0})
p = Point()
print(p.x, type(p).__name__)
```

</details>

In [ ]:
# your code here


### Q03 · Real-life

A calculator box receives text typed by users. Use `eval` with its
builtins switched off (`{"__builtins__": {}}` as globals) to compute each
expression and print the result. When evaluating raises an exception, print
`blocked:` and the exception's class name instead.

**Expected output:**

```text
14
2.0
blocked: NameError
```

<details><summary><b>Solution</b></summary>

```python
for text in exprs:
    try:
        print(eval(text, {"__builtins__": {}}))
    except Exception as err:
        print("blocked:", type(err).__name__)
```

</details>

In [ ]:
exprs = ["2 + 3 * 4", "(10 - 4) / 3", "__import__('os').getcwd()"]

# your code here


### Q04 · Real-life

A shop class charges 20% tax through its `tax_rate(self)` method. For a
one-off sale test, replace that method **on the class** (monkey-patch)
with a function that returns `0.0`, then print the rate of a new shop.

**Expected output:**

```text
0.2
0.0
```

<details><summary><b>Solution</b></summary>

```python
Shop.tax_rate = lambda self: 0.0
print(Shop().tax_rate())
```

</details>

In [ ]:
class Shop:
    def tax_rate(self):
        return 0.2

print(Shop().tax_rate())

# your code here


### Q05 · Build

Using `inspect` and `vars`: print the list of parameter names of the
`Order` constructor, then print the instance dictionary of
`Order("tea", 2)`.

**Expected output:**

```text
['item', 'qty']
{'item': 'tea', 'qty': 2}
```

<details><summary><b>Solution</b></summary>

```python
import inspect
print(list(inspect.signature(Order).parameters))
print(vars(Order("tea", 2)))
```

</details>

In [ ]:
class Order:
    def __init__(self, item, qty=1):
        self.item = item
        self.qty = qty

# your code here


---

## Combine · 2 to 4 lines

### Q06 · Build

Write a descriptor class `Tag` whose `__set_name__` prints
`bound to <attribute name>`. Use it as two class attributes `color` and
`wheels` of a `Car` class and just run the class definition.

**Expected output:**

```text
bound to color
bound to wheels
```

<details><summary><b>Solution</b></summary>

```python
class Tag:
    def __set_name__(self, owner, name):
        print("bound to", name)

class Car:
    color = Tag()
    wheels = Tag()
```

</details>

In [ ]:
# your code here


### Q07 · Real-life

Build a plugin list with `__init_subclass__`. The base class `Plugin` keeps
a list `names` and every subclass adds its own class name when it is
defined. Define `Backup` and `Cleanup` as subclasses, then print the list.

**Expected output:**

```text
['Backup', 'Cleanup']
```

<details><summary><b>Solution</b></summary>

```python
class Plugin:
    names = []
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin.names.append(cls.__name__)

class Backup(Plugin): pass
class Cleanup(Plugin): pass
print(Plugin.names)
```

</details>

In [ ]:
# your code here


### Q08 · Output-match

`property` is built on the descriptor protocol. For the class below, print
the type name of what is stored under `"celsius"` in the class dictionary,
then whether that object has `__get__` and whether it has `__set__`.

**Expected output:**

```text
property
True True
```

<details><summary><b>Solution</b></summary>

```python
prop = Temp.__dict__["celsius"]
print(type(prop).__name__)
print(hasattr(prop, "__get__"), hasattr(prop, "__set__"))
```

</details>

In [ ]:
class Temp:
    @property
    def celsius(self):
        return 21.5

# your code here


### Q09 · Puzzle

Create a class `Demo` with attribute `n` set to a `NonData()` and
attribute `d` set to a `Data()`. Make an instance and assign the text
`"mine"` to both `n` and `d`. Print `n`, then `d`, and see which one the
instance really kept.

**Expected output:**

```text
mine
descriptor
```

<details><summary><b>Solution</b></summary>

```python
class Demo:
    n = NonData()
    d = Data()

demo = Demo()
demo.n = "mine"
demo.d = "mine"
print(demo.n)
print(demo.d)
```

</details>

In [ ]:
class NonData:
    def __get__(self, obj, owner=None):
        return "descriptor"

class Data(NonData):
    def __set__(self, obj, value):
        pass

# your code here


### Q10 · Output-match

Write a class `Box` with a `__class_getitem__` so that `Box[int]` returns
the text `Box of int` (use the type's `__name__`). Print `Box[int]` and
`Box[str]`.

**Expected output:**

```text
Box of int
Box of str
```

<details><summary><b>Solution</b></summary>

```python
class Box:
    def __class_getitem__(cls, item):
        return "Box of " + item.__name__

print(Box[int])
print(Box[str])
```

</details>

In [ ]:
# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Real-life

Write a descriptor `Typed(expected_type)` that stores the value in the
instance's `__dict__` under the attribute's own name (use `__set_name__`)
and raises `TypeError` when the value has the wrong type. Use it for `name`
(str) and `age` (int) in an `Employee` class whose `__init__` takes both.
Create `Employee("Mia", 31)` and print its name and age, then try
`Employee("Sam", "forty")` and print the error message.

**Expected output:**

```text
Mia 31
age must be int
```

<details><summary><b>Solution</b></summary>

```python
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, owner=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]
    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} must be {self.expected_type.__name__}")
        obj.__dict__[self.name] = value

class Employee:
    name = Typed(str)
    age = Typed(int)
    def __init__(self, name, age):
        self.name = name
        self.age = age

mia = Employee("Mia", 31)
print(mia.name, mia.age)
try:
    Employee("Sam", "forty")
except TypeError as err:
    print(err)
```

</details>

In [ ]:
# your code here


### Q12 · Real-life

Write an `Audited` class that records the name of every attribute read from
an instance in a list `reads` (stored in the instance dictionary), by
overriding `__getattribute__`. Take care not to recurse forever. Set
`price` and `stock` in `__init__`, read `price`, `stock` and `price` again,
and print the log of names.

**Expected output:**

```text
['price', 'stock', 'price']
```

<details><summary><b>Solution</b></summary>

```python
class Audited:
    def __init__(self):
        self.__dict__["reads"] = []
        self.price = 9.5
        self.stock = 12
    def __getattribute__(self, name):
        if name in ("price", "stock"):
            object.__getattribute__(self, "reads").append(name)
        return object.__getattribute__(self, name)

item = Audited()
item.price
item.stock
item.price
print(item.reads)
```

</details>

In [ ]:
# your code here


### Q13 · Puzzle

See how `classmethod` works by building your own. Write a descriptor class
`MyClassMethod` that wraps a function, and whose `__get__` returns a
callable that passes the **owner class** as the first argument (a lambda
or a small inner function works). Use it on `Pizza.from_text("margherita:8")`,
which should return a `Pizza` with a name and a price, then print its
name and price as a number.

**Expected output:**

```text
margherita 8
```

<details><summary><b>Solution</b></summary>

```python
class MyClassMethod:
    def __init__(self, func):
        self.func = func
    def __get__(self, obj, owner=None):
        return lambda *args: self.func(owner, *args)

class Pizza:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    @MyClassMethod
    def from_text(cls, text):
        name, price = text.split(":")
        return cls(name, int(price))

pizza = Pizza.from_text("margherita:8")
print(pizza.name, pizza.price)
```

</details>

In [ ]:
# your code here


### Q14 · Real-life

Export formats register themselves. Write a base class `Exporter` with a
dict `registry`; `__init_subclass__(cls, fmt, **kwargs)` stores the
subclass under `fmt`. Create `CsvExporter` (`fmt="csv"`, `run(rows)`
returns the rows joined by commas) and `LinesExporter` (`fmt="lines"`,
joined by newlines). Write `export(fmt, rows)` that looks up the class,
creates it and returns its output, then print both formats for `rows`.

**Expected output:**

```text
tea,milk,eggs
tea
milk
eggs
```

<details><summary><b>Solution</b></summary>

```python
class Exporter:
    registry = {}
    def __init_subclass__(cls, fmt, **kwargs):
        super().__init_subclass__(**kwargs)
        Exporter.registry[fmt] = cls

class CsvExporter(Exporter, fmt="csv"):
    def run(self, rows):
        return ",".join(rows)

class LinesExporter(Exporter, fmt="lines"):
    def run(self, rows):
        return "\n".join(rows)

def export(fmt, rows):
    return Exporter.registry[fmt]().run(rows)

print(export("csv", rows))
print(export("lines", rows))
```

</details>

In [ ]:
rows = ["tea", "milk", "eggs"]

# your code here


### Q15 · Real-life

Generate data classes from a schema, using `type()` and no `class`
keyword. Write `make_model(name, fields)` that returns a new class whose
`__init__` takes one positional argument per field name and stores it, and
whose `__repr__` returns `Name(field=value, ...)` in field order. Build all
models in `schema` and print one instance of each.

**Expected output:**

```text
Dog(name='Rex', age=4)
Book(title='Dune', pages=412, author='Herbert')
```

<details><summary><b>Solution</b></summary>

```python
def make_model(name, fields):
    def __init__(self, *values):
        for field, value in zip(fields, values):
            setattr(self, field, value)
    def __repr__(self):
        parts = [f"{f}={getattr(self, f)!r}" for f in fields]
        return f"{name}({', '.join(parts)})"
    return type(name, (), {"__init__": __init__, "__repr__": __repr__})

models = {n: make_model(n, f) for n, f in schema.items()}
print(models["Dog"]("Rex", 4))
print(models["Book"]("Dune", 412, "Herbert"))
```

</details>

In [ ]:
schema = {"Dog": ["name", "age"], "Book": ["title", "pages", "author"]}

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Real-life

Build reusable validated fields. A base descriptor `Field` stores values in
the instance `__dict__` (use `__set_name__`) and calls `self.check(value)`
before storing. `IntField(low, high)` raises `ValueError` with the message
`<name> must be between <low> and <high>` when out of range, and
`TextField(max_len)` raises `ValueError` with `<name> is too long` when the
text is longer. Use them in a `Product` class (`title` TextField 10,
`stock` IntField 0 to 100), then try three products and print either
`ok` and the title, or the error message for each.

**Expected output:**

```text
ok Tea
title is too long
stock must be between 0 and 100
```

<details><summary><b>Solution</b></summary>

```python
class Field:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, owner=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]
    def __set__(self, obj, value):
        self.check(value)
        obj.__dict__[self.name] = value

class IntField(Field):
    def __init__(self, low, high):
        self.low, self.high = low, high
    def check(self, value):
        if not self.low <= value <= self.high:
            raise ValueError(f"{self.name} must be between {self.low} and {self.high}")

class TextField(Field):
    def __init__(self, max_len):
        self.max_len = max_len
    def check(self, value):
        if len(value) > self.max_len:
            raise ValueError(f"{self.name} is too long")

class Product:
    title = TextField(10)
    stock = IntField(0, 100)
    def __init__(self, title, stock):
        self.title = title
        self.stock = stock

for title, stock in attempts:
    try:
        product = Product(title, stock)
        print("ok", product.title)
    except ValueError as err:
        print(err)
```

</details>

In [ ]:
attempts = [("Tea", 20), ("Extra long title", 5), ("Milk", 500)]

# your code here


### Q17 · Puzzle

Write your own `cached_property` as a **non-data** descriptor: its
`__get__` calls the function, stores the result in the instance
`__dict__` under the same name, and returns it, so the next read never
reaches the descriptor again. Use it on `Report.total` (sums the `sales`
list given to the report, and counts every real computation in a class
attribute `computed`). Read `total` three times and print the total and the
number of computations.

**Expected output:**

```text
245 1
```

<details><summary><b>Solution</b></summary>

```python
class cached:
    def __init__(self, func):
        self.func = func
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, owner=None):
        if obj is None:
            return self
        value = self.func(obj)
        obj.__dict__[self.name] = value
        return value

class Report:
    computed = 0
    def __init__(self, sales):
        self.sales = sales
    @cached
    def total(self):
        Report.computed += 1
        return sum(self.sales)

report = Report([120, 80, 45])
report.total
report.total
print(report.total, Report.computed)
```

</details>

In [ ]:
# your code here


### Q18 · Real-life

A game needs exactly one `Settings` object. Write a metaclass `Singleton`
(a subclass of `type`) that keeps a dict of instances and overrides
`__call__`, so calling the class again returns the first instance. `Settings`
counts in a class attribute `created` how many times `__init__` really ran.
Create it three times, then print whether all three are the same object
and the value of `created`.

**Expected output:**

```text
True
1
```

<details><summary><b>Solution</b></summary>

```python
class Singleton(type):
    instances = {}
    def __call__(cls, *args, **kwargs):
        if cls not in Singleton.instances:
            Singleton.instances[cls] = super().__call__(*args, **kwargs)
        return Singleton.instances[cls]

class Settings(metaclass=Singleton):
    created = 0
    def __init__(self):
        Settings.created += 1

a = Settings()
b = Settings()
c = Settings()
print(a is b is c)
print(Settings.created)
```

</details>

In [ ]:
# your code here


### Q19 · Puzzle

Customise `isinstance`. Write an abstract class `Sized2` (from `abc.ABC`)
with a `__subclasshook__` classmethod that says any class defining
`__len__` counts as a subclass (return `NotImplemented` otherwise). Print
`isinstance` for `[1, 2]`, `"hi"` and `42`. Then make `Ticket`, a plain
class with no `__len__`, count as well by using `Sized2.register(Ticket)`,
and print `isinstance(Ticket(), Sized2)`.

**Expected output:**

```text
True True False
True
```

<details><summary><b>Solution</b></summary>

```python
from abc import ABC

class Sized2(ABC):
    @classmethod
    def __subclasshook__(cls, other):
        if cls is Sized2 and hasattr(other, "__len__"):
            return True
        return NotImplemented

class Ticket: pass

print(isinstance([1, 2], Sized2), isinstance("hi", Sized2), isinstance(42, Sized2))
Sized2.register(Ticket)
print(isinstance(Ticket(), Sized2))
```

</details>

In [ ]:
# your code here


---

## Q20 · Story project: Mini ORM

Your team wants a tiny in-memory ORM so the shop's data classes stop
repeating validation code. You'll build it in layers: typed field
descriptors, a model base class that discovers its fields and registers
itself, a generated `repr`, and simple saving and querying. Each step uses
the classes from the step before.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
raw_rows = [
    {"table": "Customer", "name": "Mia", "age": 31},
    {"table": "Customer", "name": "Leo", "age": 17},
    {"table": "Item", "name": "Tea", "price": 4},
]


### Step 1

Write a descriptor base class `Field(kind)` that stores the value in the instance `__dict__` under its own name (`__set_name__`) and raises `TypeError` with the message `<name> expects <kind name>` for a wrong type. Define `IntField()` and `StrField()` as subclasses that fix the kind to `int` / `str`. Test them on a small throwaway class: set a good value and print it, then set a wrong one and print the error.

**Expected output:**

```text
ok
label expects str
```

<details><summary><b>Solution</b></summary>

```python
class Field:
    def __init__(self, kind):
        self.kind = kind
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, owner=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)
    def __set__(self, obj, value):
        if not isinstance(value, self.kind):
            raise TypeError(f"{self.name} expects {self.kind.__name__}")
        obj.__dict__[self.name] = value

class IntField(Field):
    def __init__(self):
        super().__init__(int)

class StrField(Field):
    def __init__(self):
        super().__init__(str)

class Demo:
    label = StrField()

demo = Demo()
demo.label = "ok"
print(demo.label)
try:
    demo.label = 5
except TypeError as err:
    print(err)
```

</details>

In [ ]:
# your code here


### Step 2

Write a base class `Model`. Its `__init_subclass__` collects every `Field` in the new class into a dict `fields` (name to descriptor, in definition order) and stores the class in `Model.registry` under its class name. `__init__(self, **values)` sets each given value through the descriptors. Define `Customer` (`name` StrField, `age` IntField) and `Item` (`name` StrField, `price` IntField). Print the registry's sorted keys and the field names of `Customer`.

**Expected output:**

```text
['Customer', 'Item']
['name', 'age']
```

<details><summary><b>Solution</b></summary>

```python
class Model:
    registry = {}
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.fields = {n: f for n, f in vars(cls).items() if isinstance(f, Field)}
        Model.registry[cls.__name__] = cls
    def __init__(self, **values):
        for key, value in values.items():
            setattr(self, key, value)

class Customer(Model):
    name = StrField()
    age = IntField()

class Item(Model):
    name = StrField()
    price = IntField()

print(sorted(Model.registry))
print(list(Customer.fields))
```

</details>

In [ ]:
# your code here


### Step 3

Attach a generated `__repr__` to `Model` (write a function and assign it): it should return `ClassName(field=value, ...)` using the class's `fields`, in order, with `!r` on the values. Then build every row of `raw_rows` with the registry (look the class up by the row's `"table"` key, pass the other keys as values) and print each object.

**Expected output:**

```text
Customer(name='Mia', age=31)
Customer(name='Leo', age=17)
Item(name='Tea', price=4)
```

<details><summary><b>Solution</b></summary>

```python
def model_repr(self):
    parts = [f"{n}={getattr(self, n)!r}" for n in self.fields]
    return f"{type(self).__name__}({', '.join(parts)})"

Model.__repr__ = model_repr
objects = []
for row in raw_rows:
    values = {k: v for k, v in row.items() if k != "table"}
    objects.append(Model.registry[row["table"]](**values))
for obj in objects:
    print(obj)
```

</details>

In [ ]:
# your code here


### Step 4

Add storage by attaching to `Model`: a dict `tables` mapping a class name to a list of saved objects, a method `save(self)` that appends the object to its class's list, and a classmethod `where(cls, **conditions)` returning the saved objects whose attributes all match the conditions (no conditions means all). Save every object, then print the names of customers with `age` 31 and the number of saved items.

**Expected output:**

```text
['Mia']
1
```

<details><summary><b>Solution</b></summary>

```python
Model.tables = {}

def save(self):
    Model.tables.setdefault(type(self).__name__, []).append(self)

def where(cls, **conditions):
    found = []
    for obj in Model.tables.get(cls.__name__, []):
        if all(getattr(obj, k) == v for k, v in conditions.items()):
            found.append(obj)
    return found

Model.save = save
Model.where = classmethod(where)
for obj in objects:
    obj.save()
print([c.name for c in Customer.where(age=31)])
print(len(Item.where()))
```

</details>

In [ ]:
# your code here


### Step 5

Final check: try to create `Customer(name="Zed", age="old")` and print the error message; then print the saved customers who are 18 or older, using `Customer.where()` and a list comprehension.

**Expected output:**

```text
age expects int
[Customer(name='Mia', age=31)]
```

<details><summary><b>Solution</b></summary>

```python
try:
    Customer(name="Zed", age="old")
except TypeError as err:
    print(err)
adults = [c for c in Customer.where() if c.age >= 18]
print(adults)
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB26 · Type Hints** in `advanced/`.